# Full Workforce Validation

This notebook validates the complete 10,000-employee synthetic workforce.

The full hierarchy contains:

- Department heads
- Senior managers
- Team managers
- Individual contributors

The validation focuses on:

- Table size
- Primary keys
- Foreign keys
- Employment-status rules
- Manager relationships
- Department consistency
- Direct-report limits

In [1]:
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

employees = pd.read_csv(
    RAW_DATA_DIR / "employees.csv",
    parse_dates=[
        "hire_date",
        "termination_date",
    ],
)

departments = pd.read_csv(
    RAW_DATA_DIR / "departments.csv"
)

locations = pd.read_csv(
    RAW_DATA_DIR / "locations.csv"
)

job_roles = pd.read_csv(
    RAW_DATA_DIR / "job_roles.csv"
)

employees["manager_id"] = (
    employees["manager_id"]
    .astype("Int64")
)

print(employees.shape)

(10000, 15)


## 1. Organizational-level counts

In [2]:
organizational_level_counts = (
    employees["organizational_level"]
    .value_counts()
    .rename_axis("organizational_level")
    .reset_index(name="employee_count")
)

organizational_level_counts

,organizational_level,employee_count
0,Individual Contributor,9158
1,Team Manager,766
2,Senior Manager,68
3,Department Head,8


## 2. Department hierarchy

In [3]:
hierarchy_by_department = pd.crosstab(
    employees["department_id"],
    employees["organizational_level"],
)

expected_columns = [
    "Department Head",
    "Senior Manager",
    "Team Manager",
    "Individual Contributor",
]

hierarchy_by_department = (
    hierarchy_by_department
    .reindex(
        columns=expected_columns,
        fill_value=0,
    )
    .reset_index()
    .merge(
        departments[
            [
                "department_id",
                "department_name",
            ]
        ],
        on="department_id",
        how="left",
    )
)

hierarchy_by_department[
    "total_employees"
] = (
    hierarchy_by_department[
        expected_columns
    ].sum(axis=1)
)

hierarchy_by_department[
    [
        "department_id",
        "department_name",
        "Department Head",
        "Senior Manager",
        "Team Manager",
        "Individual Contributor",
        "total_employees",
    ]
]

,department_id,department_name,Department Head,Senior Manager,Team Manager,Individual Contributor,total_employees
0,1,Engineering,1,13,153,1833,2000
1,2,Manufacturing,1,16,191,2291,2499
2,3,Supply Chain,1,8,92,1099,1200
3,4,Sales,1,7,77,915,1000
4,5,Finance,1,6,61,732,800
5,6,Human Resources,1,5,54,641,701
6,7,Information Technology,1,7,77,915,1000
7,8,Customer Support,1,6,61,732,800


In [4]:
## 3. Manager relationships

In [5]:
employee_lookup = (
    employees
    .set_index("employee_id")
)

managed_employees = employees[
    employees["manager_id"].notna()
].copy()

managed_employees[
    "manager_department_id"
] = (
    managed_employees["manager_id"]
    .astype(int)
    .map(
        employee_lookup[
            "department_id"
        ]
    )
)

managed_employees[
    "manager_status"
] = (
    managed_employees["manager_id"]
    .astype(int)
    .map(
        employee_lookup[
            "employment_status"
        ]
    )
)

managed_employees[
    "manager_level"
] = (
    managed_employees["manager_id"]
    .astype(int)
    .map(
        employee_lookup[
            "organizational_level"
        ]
    )
)

managed_employees[
    [
        "employee_id",
        "organizational_level",
        "department_id",
        "manager_id",
        "manager_level",
        "manager_department_id",
        "manager_status",
    ]
].head(20)

,employee_id,organizational_level,department_id,manager_id,manager_level,manager_department_id,manager_status
1,100002,Senior Manager,1,100001,Department Head,1,Active
2,100003,Senior Manager,1,100001,Department Head,1,Active
3,100004,Senior Manager,1,100001,Department Head,1,Active
4,100005,Senior Manager,1,100001,Department Head,1,Active
5,100006,Senior Manager,1,100001,Department Head,1,Active
6,100007,Senior Manager,1,100001,Department Head,1,Active
7,100008,Senior Manager,1,100001,Department Head,1,Active
8,100009,Senior Manager,1,100001,Department Head,1,Active
9,100010,Senior Manager,1,100001,Department Head,1,Active
10,100011,Senior Manager,1,100001,Department Head,1,Active


## 4. Direct-report counts

In [6]:
direct_report_counts = (
    employees["manager_id"]
    .dropna()
    .astype(int)
    .value_counts()
    .rename_axis("manager_id")
    .reset_index(name="direct_reports")
)

manager_summary = (
    direct_report_counts
    .merge(
        employees[
            [
                "employee_id",
                "first_name",
                "last_name",
                "department_id",
                "organizational_level",
            ]
        ],
        left_on="manager_id",
        right_on="employee_id",
        how="left",
    )
    .merge(
        departments[
            [
                "department_id",
                "department_name",
            ]
        ],
        on="department_id",
        how="left",
    )
)

manager_summary[
    [
        "manager_id",
        "first_name",
        "last_name",
        "department_name",
        "organizational_level",
        "direct_reports",
    ]
].sort_values(
    "direct_reports",
    ascending=False,
).head(20)

,manager_id,first_name,last_name,department_name,organizational_level,direct_reports
0,102001,David,Olson,Manufacturing,Department Head,16
1,100001,Danielle,Johnson,Engineering,Department Head,13
2,100002,Joshua,Walker,Engineering,Senior Manager,12
3,100003,Jill,Rhodes,Engineering,Senior Manager,12
4,100004,Patricia,Miller,Engineering,Senior Manager,12
5,100005,Robert,Johnson,Engineering,Senior Manager,12
6,100006,Jeffery,Wagner,Engineering,Senior Manager,12
7,100007,Anthony,Gonzalez,Engineering,Senior Manager,12
8,100008,Debra,Gardner,Engineering,Senior Manager,12
9,100009,Jeffrey,Lawrence,Engineering,Senior Manager,12


In [7]:
direct_reports_by_level = (
    manager_summary
    .groupby("organizational_level")
    .agg(
        manager_count=(
            "manager_id",
            "count",
        ),
        minimum_direct_reports=(
            "direct_reports",
            "min",
        ),
        average_direct_reports=(
            "direct_reports",
            "mean",
        ),
        maximum_direct_reports=(
            "direct_reports",
            "max",
        ),
    )
    .round(2)
)

direct_reports_by_level

,manager_count,minimum_direct_reports,average_direct_reports,maximum_direct_reports
organizational_level,,,,
Department Head,8,5,8.50,16
Senior Manager,68,10,11.26,12
Team Manager,766,11,11.96,12


## 5. Complete validation checks

In [8]:
valid_employee_ids = set(
    employees["employee_id"]
)

valid_department_ids = set(
    departments["department_id"]
)

valid_location_ids = set(
    locations["location_id"]
)

valid_job_role_ids = set(
    job_roles["job_role_id"]
)

used_manager_ids = set(
    employees["manager_id"]
    .dropna()
    .astype(int)
)

active_mask = (
    employees["employment_status"]
    == "Active"
)

terminated_mask = (
    employees["employment_status"]
    == "Terminated"
)

department_heads = employees[
    employees["organizational_level"]
    == "Department Head"
]

senior_managers = managed_employees[
    managed_employees["organizational_level"]
    == "Senior Manager"
]

team_managers = managed_employees[
    managed_employees["organizational_level"]
    == "Team Manager"
]

individual_contributors = managed_employees[
    managed_employees["organizational_level"]
    == "Individual Contributor"
]

department_head_limit_passed = (
    manager_summary.loc[
        manager_summary[
            "organizational_level"
        ]
        == "Department Head",
        "direct_reports",
    ]
    .le(20)
    .all()
)

other_manager_limit_passed = (
    manager_summary.loc[
        manager_summary[
            "organizational_level"
        ].isin(
            [
                "Senior Manager",
                "Team Manager",
            ]
        ),
        "direct_reports",
    ]
    .le(12)
    .all()
)

validation_checks = pd.Series(
    {
        "table has 10,000 rows": (
            len(employees) == 10_000
        ),
        "table has 15 columns": (
            len(employees.columns) == 15
        ),
        "employee IDs are complete": (
            employees["employee_id"]
            .notna()
            .all()
        ),
        "employee IDs are unique": (
            employees["employee_id"]
            .is_unique
        ),
        "department IDs are valid": (
            set(employees["department_id"])
            .issubset(valid_department_ids)
        ),
        "location IDs are valid": (
            set(employees["location_id"])
            .issubset(valid_location_ids)
        ),
        "job-role IDs are valid": (
            set(employees["job_role_id"])
            .issubset(valid_job_role_ids)
        ),
        "manager IDs are valid": (
            used_manager_ids
            .issubset(valid_employee_ids)
        ),
        "no employee manages themselves": (
            not (
                employees["manager_id"].notna()
                & (
                    employees["manager_id"]
                    == employees["employee_id"]
                )
            ).any()
        ),
        "there are eight department heads": (
            len(department_heads) == 8
        ),
        "department heads have no managers": (
            department_heads[
                "manager_id"
            ].isna().all()
        ),
        "senior managers report to heads": (
            senior_managers[
                "manager_level"
            ].eq(
                "Department Head"
            ).all()
        ),
        "team managers have valid managers": (
            team_managers[
                "manager_level"
            ].isin(
                [
                    "Department Head",
                    "Senior Manager",
                ]
            ).all()
        ),
        "individual contributors have valid managers": (
            individual_contributors[
                "manager_level"
            ].isin(
                [
                    "Department Head",
                    "Team Manager",
                ]
            ).all()
        ),
        "employees and managers share departments": (
            managed_employees[
                "department_id"
            ].eq(
                managed_employees[
                    "manager_department_id"
                ]
            ).all()
        ),
        "all referenced managers are active": (
            managed_employees[
                "manager_status"
            ].eq("Active").all()
        ),
        "department heads have at most 20 reports": (
            department_head_limit_passed
        ),
        "other managers have at most 12 reports": (
            other_manager_limit_passed
        ),
        "active employees have no termination date": (
            employees.loc[
                active_mask,
                "termination_date",
            ].isna().all()
        ),
        "terminated employees have termination dates": (
            employees.loc[
                terminated_mask,
                "termination_date",
            ].notna().all()
        ),
        "terminated employees have termination types": (
            employees.loc[
                terminated_mask,
                "termination_type",
            ].notna().all()
        ),
    },
    name="passed",
)

validation_results = pd.DataFrame(
    {
        "check": validation_checks.index,
        "passed": validation_checks.values,
    }
)

validation_results

,check,passed
0,"table has 10,000 rows",True
1,table has 15 columns,True
2,employee IDs are complete,True
3,employee IDs are unique,True
4,department IDs are valid,True
5,location IDs are valid,True
6,job-role IDs are valid,True
7,manager IDs are valid,True
8,no employee manages themselves,True
9,there are eight department heads,True


## 6. Workforce summary

In [9]:
AS_OF_DATE = pd.Timestamp(
    "2026-06-30"
)

employee_end_date = (
    employees["termination_date"]
    .fillna(AS_OF_DATE)
)

employees["tenure_years"] = (
    (
        employee_end_date
        - employees["hire_date"]
    ).dt.days
    / 365.25
).round(2)

workforce_summary = pd.Series(
    {
        "total employee records": (
            len(employees)
        ),
        "active employees": (
            employees[
                "employment_status"
            ].eq("Active").sum()
        ),
        "terminated employees": (
            employees[
                "employment_status"
            ].eq("Terminated").sum()
        ),
        "voluntary terminations": (
            employees[
                "termination_type"
            ].eq("Voluntary").sum()
        ),
        "involuntary terminations": (
            employees[
                "termination_type"
            ].eq("Involuntary").sum()
        ),
        "average tenure years": (
            employees[
                "tenure_years"
            ].mean().round(2)
        ),
        "number of managers": (
            employees[
                "organizational_level"
            ].isin(
                [
                    "Department Head",
                    "Senior Manager",
                    "Team Manager",
                ]
            ).sum()
        ),
    },
    name="value",
)

workforce_summary

total employee records      10000.00
active employees             8287.00
terminated employees         1713.00
voluntary terminations       1299.00
involuntary terminations      414.00
average tenure years            2.57
number of managers            842.00
Name: value, dtype: float64

## 7. Conclusions

The full synthetic workforce contains exactly 10,000 employee records.

### Hierarchy results

- Every department has one department head.
- Large departments contain senior managers.
- Senior managers report to department heads.
- Team managers report to senior managers or department heads.
- Individual contributors report to team managers or department heads.
- Employees and managers belong to the same department.
- All referenced managers are active.
- Department heads have no more than 20 direct reports.
- Senior managers and team managers have no more than 12 direct reports.

### Data-quality results

- Employee IDs are complete and unique.
- Foreign-key values are valid.
- Active employees have no termination information.
- Terminated employees have termination dates and termination types.
- All full-workforce validation checks passed.

### Remaining limitations

- Manager titles still reuse existing departmental roles.
- Historical promotions and manager changes are not yet represented.
- Compensation, performance, training, recruiting, and employee-event records have not yet been generated.
- The current termination probabilities are intentionally simplified synthetic assumptions.